In [1]:
# ALL IMPORTS
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.base.model as Model
import joblib
import json
from itertools import combinations
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_score, validation_curve, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, log_loss
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [2]:
# CLASS DEFINITIONS
class LogisticRegressionModelBuilder:
    def __init__(self, input_train_data: pd.DataFrame, input_test_data: pd.DataFrame, independent_variables: list, target_variable: str
                 ) -> {Model, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame}:
        self.input_train_data = input_train_data
        self.input_test_data = input_test_data
        self.independent_variables = independent_variables
        self.target_variable = target_variable

    def run(self):
        self.X_train, self.y_train, self.X_test, self.y_test = self.split_data_by_independent_and_target_variables()
        self.X_train, self.X_test = self.add_constant()
        self.logistic_regression_model = self.train_logistic_regression_model()
        self.output_train_input_df, self.output_test_input_df = self.predict_target_variable()
        return self.logistic_regression_model, self.X_train, self.y_train, self.X_test, self.y_test, self.output_train_input_df, self.output_test_input_df

    def split_data_by_independent_and_target_variables(self): 
        X_train = self.input_train_data[self.independent_variables].copy()
        y_train = self.input_train_data[[self.target_variable]].copy()

        X_test = self.input_test_data[self.independent_variables].copy()
        y_test = self.input_test_data[[self.target_variable]].copy()
        return X_train, y_train, X_test, y_test
    
    def add_constant(self):
        X_train = sm.add_constant(self.X_train)
        X_test = sm.add_constant(self.X_test)
        return X_train, X_test

    def train_logistic_regression_model(self):
        #logistic_regression_model = sm.Logit(self.y_train, self.X_train).fit()
        logistic_regression_model = sm.Logit(self.y_train, self.X_train).fit(disp=False)
        #print(logistic_regression_model.summary())
        return logistic_regression_model

    def predict_target_variable(self):
        output_train_input_df = self.input_train_data.copy()
        output_test_input_df = self.input_test_data.copy()

        output_train_input_df['predicted_default_flag'] = self.logistic_regression_model.predict(self.X_train)
        output_test_input_df['predicted_default_flag'] = self.logistic_regression_model.predict(self.X_test)
        return output_train_input_df, output_test_input_df

In [3]:
class LassoLogisticRegressionBuilder:
    def __init__(self, input_train_data: pd.DataFrame, input_test_data: pd.DataFrame, 
                 independent_variables: list, target_variable: str, cv_folds: int = 5):
        self.input_train_data = input_train_data
        self.input_test_data = input_test_data
        self.independent_variables = independent_variables
        self.target_variable = target_variable
        self.cv_folds = cv_folds
        self.results_df = None
        self.best_alpha = None
        self.best_model = None
        
    def run(self):
        """Main method to run the Lasso CV analysis"""
        self.X_train, self.y_train, self.X_test, self.y_test = self.split_data_by_independent_and_target_variables()
        self.X_train_scaled, self.X_test_scaled = self.scale_features()
        self.results_df = self.perform_lasso_cv_analysis()
        self.best_alpha, self.best_model = self.get_best_model()
        self.generate_plots()
        return self.results_df, self.best_alpha, self.best_model
    
    def split_data_by_independent_and_target_variables(self):
        """Split data into features and target"""
        X_train = self.input_train_data[self.independent_variables].copy()
        y_train = self.input_train_data[self.target_variable].copy()
        
        X_test = self.input_test_data[self.independent_variables].copy()
        y_test = self.input_test_data[self.target_variable].copy()
        
        return X_train, y_train, X_test, y_test
    
    def scale_features(self):
        """Scale features for Lasso regularization"""
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(self.X_train), 
            columns=self.X_train.columns,
            index=self.X_train.index
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(self.X_test), 
            columns=self.X_test.columns,
            index=self.X_test.index
        )
        self.scaler = scaler
        return X_train_scaled, X_test_scaled
    
    def perform_lasso_cv_analysis(self):
        """Perform Lasso CV analysis across different alpha values"""
        # Define C range directly (lower C = stronger regularization)
        # C values from 10 to 0.0001 (equivalent to alpha from 0.1 to 10000)
        C_values = np.logspace(1, -4, 20)  # From 10 to 0.0001
        alphas = 1 / C_values  # Convert to alpha for display
        
        results = []
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
        
        for i, (C, alpha) in enumerate(zip(C_values, alphas)):
            print(f"Processing C={C:.6f}, alpha={alpha:.2f} ({i+1}/{len(C_values)})")
            
            # Initialize model with stronger regularization settings
            model = LogisticRegression(
                penalty='l1', 
                C=C, 
                solver='saga',  # Better for L1 regularization
                random_state=42,
                max_iter=2000,  # More iterations
                tol=1e-6  # Tighter convergence
            )
            
            # Cross-validation scores
            cv_scores = cross_val_score(
                model, self.X_train_scaled, self.y_train, 
                cv=kf, scoring='neg_log_loss'
            )
            cv_error = -cv_scores.mean()
            
            # Fit model on full training data
            model.fit(self.X_train_scaled, self.y_train)
            
            # Training error
            train_pred_proba = model.predict_proba(self.X_train_scaled)[:, 1]
            train_error = log_loss(self.y_train, train_pred_proba)
            
            # Test error (holdout error)
            test_pred_proba = model.predict_proba(self.X_test_scaled)[:, 1]
            test_error = log_loss(self.y_test, test_pred_proba)
            
            # Total error (average of train and test)
            total_error = (train_error + test_error) / 2
            
            # Number of features (non-zero coefficients with stricter threshold)
            num_features = np.sum(np.abs(model.coef_[0]) > 1e-4)  # Stricter threshold
            
            results.append({
                'Model': i + 1,
                'Alpha': alpha,
                'Num Features': num_features,
                'Train Error': train_error,
                'Test Error': test_error,
                'Total Error': total_error,
                'CV Error': cv_error
            })
        
        return pd.DataFrame(results)
    
    def get_best_model(self):
        """Get the best model based on CV error"""
        best_idx = self.results_df['CV Error'].idxmin()
        best_alpha = self.results_df.loc[best_idx, 'Alpha']
        best_C = 1 / best_alpha
        
        # Train best model
        best_model = LogisticRegression(
            penalty='l1', 
            C=best_C, 
            solver='saga', 
            random_state=42,
            max_iter=2000,
            tol=1e-6
        )
        best_model.fit(self.X_train_scaled, self.y_train)
        
        return best_alpha, best_model
    
    def generate_plots(self):
        """Generate the plots for analysis"""
        self._plot_errors_vs_alpha()
        self._plot_features_vs_alpha()
        self._plot_coefficient_paths()
    
    def _plot_errors_vs_alpha(self):
        """Plot errors vs alpha"""
        fig = go.Figure()
        
        # Add traces for different error types
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['CV Error'],
            mode='lines',
            name='CV Error',
            line=dict(color='blue', width=2)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Train Error'],
            mode='lines',
            name='Train Error',
            line=dict(color='red', dash='dash', width=2)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Test Error'],
            mode='lines',
            name='Holdout Error',
            line=dict(color='green', dash='dash', width=2)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Total Error'],
            mode='lines',
            name='Total Error',
            line=dict(color='orange', width=2)
        ))
        
        fig.update_layout(
            title='Errors vs Alpha',
            xaxis_title='Alpha',
            yaxis_title='Error',
            xaxis_type='log',
            template='plotly_white',
            height=400
        )
        
        fig.show()
    
    def _plot_features_vs_alpha(self):
        """Plot number of features vs alpha"""
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Num Features'],
            mode='lines',
            name='Number of Features',
            line=dict(color='blue', width=2)
        ))
        
        fig.update_layout(
            title='Number of Features vs Alpha',
            xaxis_title='Alpha',
            yaxis_title='Number of Features',
            xaxis_type='log',
            template='plotly_white',
            height=400
        )
        
        fig.show()
    
    def _plot_coefficient_paths(self):
        """Plot coefficient paths vs lambda (alpha)"""
        C_values_plot = np.logspace(1, -4, 50)  # More points for smoother paths
        alphas_plot = 1 / C_values_plot
        
        coefficients = []
        
        for C in C_values_plot:
            model = LogisticRegression(
                penalty='l1', 
                C=C, 
                solver='saga', 
                random_state=42,
                max_iter=2000,
                tol=1e-6
            )
            model.fit(self.X_train_scaled, self.y_train)
            coefficients.append(model.coef_[0])
        
        coefficients = np.array(coefficients)
        
        fig = go.Figure()
        
        colors = px.colors.qualitative.Plotly
        for i, feature in enumerate(self.independent_variables):
            fig.add_trace(go.Scatter(
                x=alphas_plot,
                y=coefficients[:, i],
                mode='lines',
                name=feature,
                line=dict(color=colors[i % len(colors)], width=2)
            ))
        
        fig.update_layout(
            title='Coefficients of Each Variable vs. Lambda',
            xaxis_title='Lambda (Regularization Parameter)',
            yaxis_title='Coefficient Value',
            xaxis_type='log',
            template='plotly_white',
            height=400
        )
        
        fig.show()
    
    def print_summary_table(self):
        """Print the summary table"""
        print("\nSummary Table:")
        print("=" * 80)
        display_df = self.results_df.copy()
        
        # Format the dataframe for better display
        for col in ['Train Error', 'Test Error', 'Total Error', 'CV Error']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.6f}")
        display_df['Alpha'] = display_df['Alpha'].apply(lambda x: f"{x:.2f}")
        
        print(display_df.to_string(index=False))
        
        # Print best model info
        best_idx = self.results_df['CV Error'].idxmin()
        print(f"\nBest Model (lowest CV Error):")
        print(f"Model: {self.results_df.loc[best_idx, 'Model']}")
        print(f"Alpha: {self.results_df.loc[best_idx, 'Alpha']:.6f}")
        print(f"Number of Features: {self.results_df.loc[best_idx, 'Num Features']}")
        print(f"CV Error: {self.results_df.loc[best_idx, 'CV Error']:.6f}")
    
    def get_feature_importance(self):
        """Get feature importance from the best model"""
        if self.best_model is None:
            raise ValueError("Must run the analysis first")
        
        feature_importance = pd.DataFrame({
            'Feature': self.independent_variables,
            'Coefficient': self.best_model.coef_[0],
            'Abs_Coefficient': np.abs(self.best_model.coef_[0])
        }).sort_values('Abs_Coefficient', ascending=False)
        
        return feature_importance

In [4]:
class RidgeLogisticRegressionBuilder:
    def __init__(self, input_train_data: pd.DataFrame, input_test_data: pd.DataFrame, 
                 independent_variables: list, target_variable: str, cv_folds: int = 5):
        self.input_train_data = input_train_data
        self.input_test_data = input_test_data
        self.independent_variables = independent_variables
        self.target_variable = target_variable
        self.cv_folds = cv_folds
        self.results_df = None
        self.best_alpha = None
        self.best_model = None
        
    def run(self):
        """Main method to run the Ridge CV analysis"""
        self.X_train, self.y_train, self.X_test, self.y_test = self.split_data_by_independent_and_target_variables()
        self.X_train_scaled, self.X_test_scaled = self.scale_features()
        self.results_df = self.perform_ridge_cv_analysis()
        self.best_alpha, self.best_model = self.get_best_model()
        self.generate_plots()
        return self.results_df, self.best_alpha, self.best_model
    
    def split_data_by_independent_and_target_variables(self):
        """Split data into features and target"""
        X_train = self.input_train_data[self.independent_variables].copy()
        y_train = self.input_train_data[self.target_variable].copy()
        
        X_test = self.input_test_data[self.independent_variables].copy()
        y_test = self.input_test_data[self.target_variable].copy()
        
        return X_train, y_train, X_test, y_test
    
    def scale_features(self):
        """Scale features for Ridge regularization"""
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(self.X_train), 
            columns=self.X_train.columns,
            index=self.X_train.index
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(self.X_test), 
            columns=self.X_test.columns,
            index=self.X_test.index
        )
        self.scaler = scaler
        return X_train_scaled, X_test_scaled
    
    def perform_ridge_cv_analysis(self):
        """Perform Ridge CV analysis across different alpha values"""
        # Define C range (C = 1/alpha for sklearn LogisticRegression)
        # Ridge works well with a wider range since coefficients don't go to zero
        C_values = np.logspace(2, -3, 20)  # From 100 to 0.001
        alphas = 1 / C_values  # Convert to alpha for display
        
        results = []
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
        
        for i, (C, alpha) in enumerate(zip(C_values, alphas)):
            print(f"Processing C={C:.6f}, alpha={alpha:.2f} ({i+1}/{len(C_values)})")
            
            # Initialize Ridge model (L2 penalty)
            model = LogisticRegression(
                penalty='l2', 
                C=C, 
                solver='lbfgs',  # Good solver for L2 regularization
                random_state=42,
                max_iter=2000
            )
            
            # Cross-validation scores
            cv_scores = cross_val_score(
                model, self.X_train_scaled, self.y_train, 
                cv=kf, scoring='neg_log_loss'
            )
            cv_error = -cv_scores.mean()
            
            # Fit model on full training data
            model.fit(self.X_train_scaled, self.y_train)
            
            # Training error
            train_pred_proba = model.predict_proba(self.X_train_scaled)[:, 1]
            train_error = log_loss(self.y_train, train_pred_proba)
            
            # Test error (holdout error)
            test_pred_proba = model.predict_proba(self.X_test_scaled)[:, 1]
            test_error = log_loss(self.y_test, test_pred_proba)
            
            # Total error (average of train and test)
            total_error = (train_error + test_error) / 2
            
            # For Ridge, all features are always retained (num_features = 9)
            num_features = len(self.independent_variables)  # Always 9 for Ridge
            
            results.append({
                'Model': i + 1,
                'Alpha': alpha,
                'Num Features': num_features,
                'Train Error': train_error,
                'Test Error': test_error,
                'Total Error': total_error,
                'CV Error': cv_error
            })
        
        return pd.DataFrame(results)
    
    def get_best_model(self):
        """Get the best model based on CV error"""
        best_idx = self.results_df['CV Error'].idxmin()
        best_alpha = self.results_df.loc[best_idx, 'Alpha']
        best_C = 1 / best_alpha
        
        # Train best model
        best_model = LogisticRegression(
            penalty='l2', 
            C=best_C, 
            solver='lbfgs', 
            random_state=42,
            max_iter=2000
        )
        best_model.fit(self.X_train_scaled, self.y_train)
        
        return best_alpha, best_model
    
    def generate_plots(self):
        """Generate the plots for analysis"""
        self._plot_errors_vs_alpha()
        self._plot_coefficient_paths()
    
    def _plot_errors_vs_alpha(self):
        """Plot errors vs alpha"""
        fig = go.Figure()
        
        # Add traces for different error types
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['CV Error'],
            mode='lines+markers',
            name='CV Error',
            line=dict(color='blue', width=2),
            marker=dict(size=6)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Train Error'],
            mode='lines+markers',
            name='Train Error',
            line=dict(color='red', dash='dash', width=2),
            marker=dict(size=6)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Test Error'],
            mode='lines+markers',
            name='Holdout Error',
            line=dict(color='green', dash='dash', width=2),
            marker=dict(size=6)
        ))
        
        fig.add_trace(go.Scatter(
            x=self.results_df['Alpha'],
            y=self.results_df['Total Error'],
            mode='lines+markers',
            name='Total Error',
            line=dict(color='orange', width=2),
            marker=dict(size=6)
        ))
        
        # Mark the best alpha
        best_idx = self.results_df['CV Error'].idxmin()
        best_alpha = self.results_df.loc[best_idx, 'Alpha']
        best_cv_error = self.results_df.loc[best_idx, 'CV Error']
        
        fig.add_trace(go.Scatter(
            x=[best_alpha],
            y=[best_cv_error],
            mode='markers',
            name=f'Best Alpha ({best_alpha:.2f})',
            marker=dict(color='black', size=12, symbol='star')
        ))
        
        fig.update_layout(
            title='Errors vs Alpha (Ridge Regression)',
            xaxis_title='Alpha',
            yaxis_title='Error',
            xaxis_type='log',
            template='plotly_white',
            height=500
        )
        
        fig.show()
    
    def _plot_coefficient_paths(self):
        """Plot coefficient paths vs alpha - Ridge style"""
        C_values_plot = np.logspace(2, -3, 100)  # More points for smoother paths
        alphas_plot = 1 / C_values_plot
        
        coefficients = []
        
        for C in C_values_plot:
            model = LogisticRegression(
                penalty='l2', 
                C=C, 
                solver='lbfgs', 
                random_state=42,
                max_iter=2000
            )
            model.fit(self.X_train_scaled, self.y_train)
            coefficients.append(model.coef_[0])
        
        coefficients = np.array(coefficients)
        
        fig = go.Figure()
        
        # Use distinct colors for better visibility
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
        
        for i, feature in enumerate(self.independent_variables):
            fig.add_trace(go.Scatter(
                x=alphas_plot,
                y=coefficients[:, i],
                mode='lines',
                name=feature.replace('_woe', ''),  # Clean feature names
                line=dict(color=colors[i % len(colors)], width=2.5),
                showlegend=True
            ))
        
        # Mark the best alpha with a vertical line
        if self.best_alpha is not None:
            fig.add_vline(
                x=self.best_alpha, 
                line_dash="dash", 
                line_color="black",
                line_width=2,
                annotation_text=f"Best α: {self.best_alpha:.2f}",
                annotation_position="top"
            )
        
        fig.update_layout(
            title='Ridge Feature Coefficients vs Regularization Alpha',
            xaxis_title='Alpha (log scale)',
            yaxis_title='Feature Coefficients',
            xaxis=dict(
                type='log',
                range=[-2, 3],  # 10^-2 to 10^3
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray'
            ),
            yaxis=dict(
                range=[-3, 3],
                showgrid=True,
                gridwidth=1,
                gridcolor='lightgray'
            ),
            plot_bgcolor='white',
            width=800,
            height=500,
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=1.01
            )
        )
        
        fig.show()
    
    def print_summary_table(self):
        """Print the summary table"""
        print("\nRidge Logistic Regression Summary Table:")
        print("=" * 80)
        display_df = self.results_df.copy()
        
        # Format the dataframe for better display
        for col in ['Train Error', 'Test Error', 'Total Error', 'CV Error']:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.6f}")
        display_df['Alpha'] = display_df['Alpha'].apply(lambda x: f"{x:.2f}")
        
        print(display_df.to_string(index=False))
        
        # Print best model info
        best_idx = self.results_df['CV Error'].idxmin()
        print(f"\nBest Ridge Model (lowest CV Error):")
        print(f"Model: {self.results_df.loc[best_idx, 'Model']}")
        print(f"Alpha: {self.results_df.loc[best_idx, 'Alpha']:.6f}")
        print(f"Number of Features: {self.results_df.loc[best_idx, 'Num Features']} (all retained in Ridge)")
        print(f"CV Error: {self.results_df.loc[best_idx, 'CV Error']:.6f}")
    
    def get_feature_importance(self):
        """Get feature importance from the best model"""
        if self.best_model is None:
            raise ValueError("Must run the analysis first")
        
        feature_importance = pd.DataFrame({
            'Feature': self.independent_variables,
            'Coefficient': self.best_model.coef_[0],
            'Abs_Coefficient': np.abs(self.best_model.coef_[0])
        }).sort_values('Abs_Coefficient', ascending=False)
        
        return feature_importance
    
    def compare_regularization_effect(self):
        """Compare the effect of different alpha values on coefficients"""
        print("\nRegularization Effect Comparison:")
        print("=" * 60)
        
        # Show coefficients for low, medium, and high regularization
        low_reg_idx = 0  # Lowest alpha (least regularization)
        high_reg_idx = len(self.results_df) - 1  # Highest alpha (most regularization)
        best_reg_idx = self.results_df['CV Error'].idxmin()  # Best alpha
        
        # Get models for comparison
        alphas_to_compare = [
            (self.results_df.loc[low_reg_idx, 'Alpha'], "Low Regularization"),
            (self.results_df.loc[best_reg_idx, 'Alpha'], "Best Alpha"),
            (self.results_df.loc[high_reg_idx, 'Alpha'], "High Regularization")
        ]
        
        comparison_df = pd.DataFrame(index=self.independent_variables)
        
        for alpha, label in alphas_to_compare:
            C = 1 / alpha
            model = LogisticRegression(penalty='l2', C=C, solver='lbfgs', random_state=42, max_iter=2000)
            model.fit(self.X_train_scaled, self.y_train)
            comparison_df[f'{label}\n(α={alpha:.2f})'] = model.coef_[0]
        
        print(comparison_df.round(4))

In [ ]:
# DATA LOADING
input_file_path = "\\train_df_woe.parquet" # add path
print(rf"Input File (Prepped Input Train Data): {input_file_path}")
train_input_df = pd.read_parquet(input_file_path)

input_file_path = "\\test_df_woe.parquet" # add path
print(rf"Input File (Prepped Input Train Data): {input_file_path}")
test_input_df = pd.read_parquet(input_file_path)

# CONFIGURATION
target_variable_str = 'default_flag'

# List of features excluding 'default_flag'
features = [
    'term_woe', 'installment_woe', 'sub_grade_woe', 'home_ownership_woe',
    'annual_inc_woe', 'verification_status_woe', 'dti_woe',
    'fico_range_high_woe', 'revol_util_woe'
]

# Generate all combinations of lengths 4, 3, and 2
all_combinations = (
    list(combinations(features, 4)) +
    list(combinations(features, 3)) +
    list(combinations(features, 2))
)

# Create the dictionary with model IDs as keys
model_id_to_independent_variables_mapping_dict = {
    f"model_{i+1}": list(combo) for i, combo in enumerate(all_combinations)
}

# LOGISTIC REGRESSION MODEL EXECUTION
# Store metrics for all models
metrics_list = []

for model_id, independent_variables_list in model_id_to_independent_variables_mapping_dict.items():
    print(f'--{model_id.replace("_", " ")}--')
    logistic_regression_model, X_train, y_train, X_test, y_test, output_train_input_df, output_test_input_df = LogisticRegressionModelBuilder(
        train_input_df, test_input_df, independent_variables_list, target_variable_str).run()

    y_true = y_test[target_variable_str]
    y_pred_prob = output_test_input_df['predicted_default_flag']
    y_pred = (y_pred_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_true, y_pred_prob)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    aic = logistic_regression_model.aic
    bic = logistic_regression_model.bic
    conf_matrix = confusion_matrix(y_true, y_pred).tolist()

    metrics_list.append({
        'model_id': model_id,
        'AUC': auc,
        'Accuracy': acc,
        'F1 Score': f1,
        'AIC': aic,
        'BIC': bic,
        'Confusion Matrix': conf_matrix
    })

# Convert to DataFrame
metrics_df = pd.DataFrame(metrics_list)

# Rank models
ranked_auc = metrics_df.sort_values(by='AUC', ascending=False)
ranked_accuracy = metrics_df.sort_values(by='Accuracy', ascending=False)
ranked_f1 = metrics_df.sort_values(by='F1 Score', ascending=False)
ranked_aic = metrics_df.sort_values(by='AIC', ascending=True)
ranked_bic = metrics_df.sort_values(by='BIC', ascending=True)

# Display top 5 models for each metric
print("Top 5 Models by AUC:\n", ranked_auc.head())
print("\nTop 5 Models by Accuracy:\n", ranked_accuracy.head())
print("\nTop 5 Models by F1 Score:\n", ranked_f1.head())
print("\nTop 5 Models by AIC (lower is better):\n", ranked_aic.head())
print("\nTop 5 Models by BIC (lower is better):\n", ranked_bic.head())

print('\n-Model:Test Data-')
print(output_test_input_df[['default_flag', 'predicted_default_flag']].agg('mean'))

print('\n--End Logistic Regression Model Build--\n')

# SAVE MODEL
output_folder_path = r"\AI_ML" #add path
output_file_path = os.path.join(output_folder_path, "logistic_model.pkl")
print(f"Output File (Logistic Regression Model): {output_file_path}")

# LASSO LOGISTIC REGRESSION EXECUTION
if __name__ == "__main__":
    # Initialize and run Lasso analysis
    lasso_builder = LassoLogisticRegressionBuilder(
        train_input_df, 
        test_input_df, 
        features, 
        target_variable_str,
        cv_folds=5
    )
    
    results_df, best_alpha, best_model = lasso_builder.run()
    lasso_builder.print_summary_table()
    
    # Get feature importance
    feature_importance = lasso_builder.get_feature_importance()
    print("\nFeature Importance (Best Model):")
    print(feature_importance)

# RIDGE LOGISTIC REGRESSION EXECUTION
if __name__ == "__main__":
    # Initialize and run Ridge analysis
    ridge_builder = RidgeLogisticRegressionBuilder(
        train_input_df, 
        test_input_df, 
        features, 
        target_variable_str,
        cv_folds=5
    )
    
    results_df, best_alpha, best_model = ridge_builder.run()
    ridge_builder.print_summary_table()
    
    # Get feature importance
    feature_importance = ridge_builder.get_feature_importance()
    print("\nFeature Importance (Best Ridge Model):")
    print(feature_importance)
    
    # Compare regularization effects
    ridge_builder.compare_regularization_effect()

Input File (Prepped Input Train Data): C:\Users\JU557VB\OneDrive - EY\Desktop\Non Chargeable\AI_ML\train_df_woe.parquet
Input File (Prepped Input Train Data): C:\Users\JU557VB\OneDrive - EY\Desktop\Non Chargeable\AI_ML\test_df_woe.parquet
--model 1--
--model 2--
--model 3--
--model 4--
--model 5--
--model 6--
--model 7--
--model 8--
--model 9--
--model 10--
--model 11--
--model 12--
--model 13--
--model 14--
--model 15--
--model 16--
--model 17--
--model 18--
--model 19--
--model 20--
--model 21--
--model 22--
--model 23--
--model 24--
--model 25--
--model 26--
--model 27--
--model 28--
--model 29--
--model 30--
--model 31--
--model 32--
--model 33--
--model 34--
--model 35--
--model 36--
--model 37--
--model 38--
--model 39--
--model 40--
--model 41--
--model 42--
--model 43--
--model 44--
--model 45--
--model 46--
--model 47--
--model 48--
--model 49--
--model 50--
--model 51--
--model 52--
--model 53--
--model 54--
--model 55--
--model 56--
--model 57--
--model 58--
--model 59--
--m


Summary Table:
 Model    Alpha  Num Features Train Error Test Error Total Error CV Error
     1     0.10             9    0.458543   0.459707    0.459125 0.458564
     2     0.18             9    0.458543   0.459707    0.459125 0.458564
     3     0.34             9    0.458543   0.459707    0.459125 0.458564
     4     0.62             9    0.458543   0.459707    0.459125 0.458564
     5     1.13             9    0.458543   0.459707    0.459125 0.458564
     6     2.07             9    0.458543   0.459707    0.459125 0.458564
     7     3.79             9    0.458543   0.459707    0.459125 0.458564
     8     6.95             9    0.458543   0.459707    0.459125 0.458564
     9    12.74             9    0.458543   0.459706    0.459124 0.458564
    10    23.36             9    0.458543   0.459706    0.459124 0.458564
    11    42.81             9    0.458543   0.459705    0.459124 0.458564
    12    78.48             9    0.458543   0.459704    0.459123 0.458564
    13   143.84       


Ridge Logistic Regression Summary Table:
 Model   Alpha  Num Features Train Error Test Error Total Error CV Error
     1    0.01             9    0.458543   0.459707    0.459125 0.458564
     2    0.02             9    0.458543   0.459707    0.459125 0.458564
     3    0.03             9    0.458543   0.459707    0.459125 0.458564
     4    0.06             9    0.458543   0.459707    0.459125 0.458564
     5    0.11             9    0.458543   0.459707    0.459125 0.458564
     6    0.21             9    0.458543   0.459707    0.459125 0.458564
     7    0.38             9    0.458543   0.459707    0.459125 0.458564
     8    0.70             9    0.458543   0.459707    0.459125 0.458564
     9    1.27             9    0.458543   0.459707    0.459125 0.458564
    10    2.34             9    0.458543   0.459707    0.459125 0.458564
    11    4.28             9    0.458543   0.459707    0.459125 0.458564
    12    7.85             9    0.458543   0.459707    0.459125 0.458564
    13   

In [ ]:
# ---- Inspect and report on the best logistic model ----
def report_statsmodels_model(model_id: str):
    # Get the feature set for this model
    features_for_model = model_id_to_independent_variables_mapping_dict[model_id]
    print(f"\n[{model_id}] Features: {features_for_model}")

    # Fit model via your existing builder
    logit_model, X_tr, y_tr, X_te, y_te, out_tr_df, out_te_df = LogisticRegressionModelBuilder(
        train_input_df, test_input_df, features_for_model, target_variable_str
    ).run()

    # ---- Coefficients table (includes intercept 'const') ----
    coef_table = pd.DataFrame({
        'feature': logit_model.params.index,
        'coef': logit_model.params.values,
        'std_err': logit_model.bse.values,
        'z_value': logit_model.tvalues.values,     # z for Logit
        'p_value': logit_model.pvalues.values
    })

    print("\n[Model 24] Coefficients (including intercept):")
    print(coef_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

    # ---- Test-set performance ----
    # Handle Series vs DataFrame for y_te
    if isinstance(y_te, pd.DataFrame):
        y_true = y_te[target_variable_str].values
    else:
        y_true = pd.Series(y_te).values

    y_prob = out_te_df['predicted_default_flag'].values
    y_pred = (y_prob >= 0.5).astype(int)

    # Metrics
    auc = roc_auc_score(y_true, y_prob)
    ll  = log_loss(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    aic  = logit_model.aic
    bic  = logit_model.bic
    cm   = confusion_matrix(y_true, y_pred).tolist()

    default_rate = float(np.mean(y_true))
    mean_pred_pd = float(np.mean(y_prob))
    calib_gap = mean_pred_pd - default_rate

    print("\n[Model 24] Performance metrics on TEST:")
    print(f"AUC:                 {auc:.6f}")
    print(f"LogLoss:             {ll:.6f}")
    print(f"Accuracy (thr=0.5):  {acc:.6f}")
    print(f"Precision (thr=0.5): {prec:.6f}")
    print(f"Recall (thr=0.5):    {rec:.6f}")
    print(f"F1 (thr=0.5):        {f1:.6f}")
    print(f"AIC:                 {aic:.6f}")
    print(f"BIC:                 {bic:.6f}")
    print(f"Default rate (test): {default_rate:.6f}")
    print(f"Mean predicted PD:   {mean_pred_pd:.6f}  (calibration gap: {calib_gap:+.6f})")

    print("\n[Model 24] Confusion matrix @0.5  ->  [[TN, FP], [FN, TP]]")
    print(cm)

# ---- Call it for model_24 ----
report_statsmodels_model('model_24')



[model_24] Features: ['term_woe', 'sub_grade_woe', 'home_ownership_woe', 'dti_woe']

[Model 24] Coefficients (including intercept):
           feature      coef  std_err     z_value  p_value
             const -1.365924 0.003383 -403.775978 0.000000
          term_woe  0.463959 0.008419   55.105349 0.000000
     sub_grade_woe  0.819613 0.005613  146.019516 0.000000
home_ownership_woe  0.975596 0.019157   50.926889 0.000000
           dti_woe  0.651025 0.010827   60.127862 0.000000

[Model 24] Performance metrics on TEST:
AUC:                 0.701299
LogLoss:             0.462326
Accuracy (thr=0.5):  0.798575
Precision (thr=0.5): 0.544269
Recall (thr=0.5):    0.054682
F1 (thr=0.5):        0.099380
AIC:                 571756.608966
BIC:                 571813.294735
Default rate (test): 0.203233
Mean predicted PD:   0.202766  (calibration gap: -0.000467)

[Model 24] Confusion matrix @0.5  ->  [[TN, FP], [FN, TP]]
[[325380, 3845], [79384, 4592]]


: 